# E1.7 · Continuous control verification

**Function E — AI Governance for Agentic Systems → Building the Governance Framework — Risk and Control**  ·  *Security of AI*

Builds on **[E1.6 · Operating vs outcome guardrails](https://spbreed.github.io/cyber-commons/lessons/E1.6.html)**.

| | |
|---|---|
| Tools used | OPA, OSCAL, GLM-4.6, Claude Haiku 4.5 |

## What this lesson is

**What it covers.** Automate one evidence package on a schedule.

**Why a security engineer needs it.** Automating judgment instead of evidence collection. The control it builds is: agent-assisted evidence collection, drift detection, exception tracking.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

A control that is verified annually is a control you know about once a year. Continuous verification is the only version of assurance that keeps up with a system whose behaviour changes between tests.

> **At CyberTravels.** A control verified once a year on a system whose prompt changed on Tuesday. Continuous verification is the only version of assurance that keeps up with CyberTravels.

## 2 · The framework

```
   annual                          continuous
   +-------------+                 +-------------------------+
   | one sample  |                 | probe on every change   |
   | one date    |      vs         | sample continuously     |
   | one signature|                | escalate on failure     |
   +-------------+                 +-------------------------+

   assurance that keeps up with a system that changes weekly
```

Continuous control verification is the operating model that follows from E1.1.

The number that matters is not how much passed once. It is **how much is
currently evidenced** — controls whose most recent test is passing *and* within
its freshness window.

Three states, and the third is the one classical GRC tooling cannot express:

- **PASS** — tested, passing, in window.
- **FAIL** — tested, failing. Honest and actionable.
- **STALE** — tested, was passing, out of window. **Not a pass.**

Plus the absence state: no evidence at all, which is different from failing and
is often the largest category in a first assessment.

## 3 · Collecting the runtime evidence, as a skill

Automating a control means something has to go and look. For the network and logging controls that underwrite every default-deny claim CyberTravels makes, that is a posture collector: egress rules, private endpoints, route tables, key policies, and whether the audit trail is not merely enabled but **delivering**. It collects; it does not conclude. This is the file in this repository:

In [ ]:
# skills/attestation/aws-runtime-posture-collector/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: aws-runtime-posture-collector
description: >-
  Snapshot a deployment's cloud network, crypto and logging posture as
  evidence for default-deny and egress controls. Use to evidence network
  isolation, to check whether private endpoints and endpoint policies are in
  place, or to record the runtime configuration an attestation depends on.
allowed-tools: Bash, Read
---

# Aws Runtime Posture Collector

**Controls:** Controls 1 and 2 — runtime posture

## What this collects

Configuration state, at a point in time, for the network and crypto boundary
around one deployment. It is evidence for other skills' verdicts rather than a
verdict in itself.

## Procedure

1. **Security groups and network ACLs.** Enumerate egress rules. Any rule
   permitting `0.0.0.0/0` outbound is an egress-open finding regardless of what
   an application-layer policy says.
2. **Private endpoints.** Record whether a private endpoint exists for each
   model and gateway service in use, and read the endpoint policy — an
   unscoped endpoint policy is an endpoint that permits any principal.
3. **Route tables.** Identify NAT and internet gateways on the deployment's
   subnets. A private endpoint does not help if a default route to an internet
   gateway remains.
4. **Key policies.** Record key policies and condition keys that scope use to a
   specific service. Unconditioned key access is a finding.
5. **Logging.** Confirm the audit trail and log destinations are enabled and
   delivering, and record the retention.

## Output contract

```json
{
  "deployment_id": "str",
  "collected_at": "str",
  "network": {
    "egress_open_findings": [{"sg": "str", "rule": "str"}],
    "private_endpoints": [{"service": "str", "present": true, "policy_scoped": true}],
    "internet_route_present": false
  },
  "crypto": {"keys": [{"id": "str", "conditioned": true}]},
  "logging": {"audit_trail_enabled": true, "log_destinations": ["str"], "retention_days": 0},
  "verdict": "PASS|PARTIAL|FAIL"
}
```

## Failure modes

- **Reading configuration and calling it enforcement.** This skill records what
  is configured. Whether traffic actually obeys it is the egress verifier's job.
- **Ignoring the route table** because a private endpoint exists.
- **Recording that logging is enabled** without checking that it is delivering.
"""

In [ ]:
# Execute the skill above, using the shared runtime rather than a copy.
import glob, importlib.util, os, sys

# Kaggle mounts an attached kernel under /kaggle/input, and it uses two
# different layouts — /kaggle/input/<slug>/ on some kernels and
# /kaggle/input/notebooks/<user>/<slug>/ on others. Both were observed on the
# same account in the same hour, so match either. The recursive glob is cheap
# here because /kaggle/input holds only what is attached; globbing the working
# tree instead cost eleven seconds a notebook.
_WHERE = (sorted(glob.glob("/kaggle/input/**/cyber-commons-skill-runtime/__script__.py",
                           recursive=True))
          + [os.path.join(p, "skills/_runtime/cyber_commons_skill_runtime.py")
             for p in (".", "..", "../..")])
_found = next((p for p in _WHERE if os.path.isfile(p)), None)
if _found is None:
    # Say what was looked for and what is actually there. "The runtime is
    # missing" on its own costs whoever hits it an afternoon.
    raise SystemExit("The shared skill runtime is missing."
                     "  looked at: " + repr(_WHERE) +
                     "  /kaggle/input holds: " +
                     repr(glob.glob("/kaggle/input/**", recursive=True)[:20]) +
                     "  cwd: " + os.getcwd() +
                     ". On Kaggle it is attached to this notebook as a "
                     "source; locally it is skills/_runtime/ in the repository.")
_spec = importlib.util.spec_from_file_location("cyber_commons_skill_runtime", _found)
cyber_commons_skill_runtime = importlib.util.module_from_spec(_spec)
sys.modules["cyber_commons_skill_runtime"] = cyber_commons_skill_runtime
_spec.loader.exec_module(cyber_commons_skill_runtime)
from cyber_commons_skill_runtime import run_skill

# Split skills/attestation/aws-runtime-posture-collector/SKILL.md into the two halves an agent uses —
# the frontmatter it routes on, and the body it follows.
meta, body = run_skill(SKILL_MD)

In [ ]:
# skills/attestation/aws-runtime-posture-collector/scripts/aws_runtime_posture_collector.py — embedded verbatim from the repository.
# This is the skill's own script, not a paraphrase of it.
#!/usr/bin/env python3
"""Compute a control posture honestly, then automate one test and watch coverage move.

This is the executable half of the `aws-runtime-posture-collector` skill: the check the
SKILL.md next to it describes, run against a synthetic CyberTravels
estate so two runs can be diffed and the result argued with.

Standard library only, and deterministic, so it runs on a Kaggle
kernel with the internet switched off.
"""

import time
from dataclasses import dataclass

now = time.time(); DAY = 86400

@dataclass
class ControlTest:
    cid: str; passed: bool; evidence: str
    tested_at: float; valid_for_days: float
    def age(self, at): return (at - self.tested_at)/DAY
    def state(self, at):
        if self.age(at) > self.valid_for_days: return "STALE"
        return "PASS" if self.passed else "FAIL"

REQUIRED = ["AC-1","AC-2","SB-1","SB-2","EV-1","EV-2","DR-1","ST-1"]
TESTS = [
 ControlTest("AC-1", True,  "act chain sample",        now -   2*DAY, 30),
 ControlTest("AC-2", True,  "delegation regression",   now -   9*DAY, 30),
 ControlTest("SB-1", True,  "egress denial log",       now -  31*DAY, 30),
 ControlTest("SB-2", True,  "approval gate screenshot",now - 120*DAY, 27),
 ControlTest("EV-1", True,  "audit sample of 50",      now -   5*DAY, 60),
 ControlTest("EV-2", True,  "expert accuracy 0.81",    now -  12*DAY, 30),
 ControlTest("DR-1", False, "drift alerting not deployed", now,       30),
]

def verify(tests, required, at):
    by = {t.cid: t for t in tests}
    rows, evidenced = [], 0
    for cid in required:
        t = by.get(cid)
        if t is None:
            rows.append({"control": cid, "state": "NO EVIDENCE", "age": None})
            continue
        st = t.state(at)
        rows.append({"control": cid, "state": st, "age": round(t.age(at), 1)})
        evidenced += st == "PASS"
    return {"required": len(required), "evidenced": evidenced,
            "coverage": round(evidenced/len(required), 3), "rows": rows}

v = verify(TESTS, REQUIRED, now)
print(f"{'control':9s}{'state':14s}{'age (days)':>12}")
print("-" * 36)
for r in v["rows"]:
    print(f"{r['control']:9s}{r['state']:14s}{str(r['age']):>12}")
print(f"\ncurrently evidenced {v['evidenced']}/{v['required']} = {v['coverage']:.0%}")

point_in_time = sum(1 for t in TESTS if t.passed)
print(f"point-in-time  : {point_in_time}/{len(REQUIRED)} = "
      f"{point_in_time/len(REQUIRED):.0%}")
print(f"continuous     : {v['evidenced']}/{v['required']} = {v['coverage']:.0%}")
stale = [r["control"] for r in v["rows"] if r["state"] == "STALE"]
none  = [r["control"] for r in v["rows"] if r["state"] == "NO EVIDENCE"]
fail  = [r["control"] for r in v["rows"] if r["state"] == "FAIL"]
print(f"\nthe gap: STALE {stale}  NO EVIDENCE {none}  FAIL {fail}")
print("Nobody did anything wrong to produce the STALE rows. Time passed.")

def automated_test(cid, run_now):
    """A control test that re-runs on a schedule writes its own evidence."""
    passed, evidence = run_now()
    return ControlTest(cid, passed, evidence, tested_at=time.time(),
                       valid_for_days=30)

def check_egress_policy():
    ALLOW = {"api.github.com"}
    attempts = ["https://api.github.com/x", "http://169.254.169.254/",
                "https://collect.example.com/x"]
    from urllib.parse import urlparse
    denied = [u for u in attempts if (urlparse(u).hostname or "") not in ALLOW]
    return len(denied) == 2, f"{len(denied)}/3 destinations denied, run automatically"

fresh = [t for t in TESTS if t.cid != "SB-1"] + [automated_test("SB-1", check_egress_policy)]
v2 = verify(fresh, REQUIRED, now)
print(f"after automating SB-1: {v2['evidenced']}/{v2['required']} = {v2['coverage']:.0%}")
print(f"   SB-1 is now {[r['state'] for r in v2['rows'] if r['control']=='SB-1'][0]}"
      f" and will stay fresh without anyone remembering")
assert v2["coverage"] > v["coverage"]

print("\nprioritise automation by how often a control goes stale:")
for t in sorted(TESTS, key=lambda t: t.valid_for_days):
    per_year = round(365 / t.valid_for_days, 1)
    print(f"   {t.cid}  window {t.valid_for_days:>3.0f}d → "
          f"{per_year:>4} manual re-tests per year")

## What you just proved

The skill loads and reports its shape, and the line to take from it is the boundary it draws: configuration is not enforcement. A private endpoint next to a route table with a NAT gateway is a recorded fact and an open path at the same time, and logging that is switched on but not delivering evidences nothing at all.

## Your turn

Automate the control with the shortest freshness window first — it is the one costing the most manual effort and going stale most often. One automated test converts an annual assertion into a live control.

---

**Next → [E1.8 · Third-party and model supply chain risk](https://spbreed.github.io/cyber-commons/lessons/E1.8.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/E1.7.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/E1.7.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*